# P&L attribution: decomposing daily MTM changes by risk factor

**Prerequisites:** notebooks **05** (pricing fundamentals) and **06** (pricing across asset classes). This notebook assumes you can construct instruments, `MarketContext`, and use the pricing pipeline.

P&L attribution answers: **"Why did my position's value change from T₀ to T₁?"** by isolating the contribution of each market factor.

## Overview

finstack supports four attribution methodologies:

| Method | How it works | Trade-offs |
|---|---|---|
| **Parallel** | Isolate each factor independently (restore T₀ for one factor, keep T₁ for the rest) | Cross-effects go to residual; factors may not sum exactly |
| **Waterfall** | Apply factors sequentially in a specified order | Guarantees sum = total P&L; order matters |
| **Metrics-based** | Linear approximation using pre-computed sensitivities (DV01, CS01, Vega, Theta…) | Fast; less accurate for large moves |
| **Taylor** | Bump-and-reprice sensitivities × observed market moves; optional second-order | Factor-additive; independent of application order |

All four return a `PnlAttribution` object that decomposes total P&L into:
- **Carry** (theta + accruals)
- **Rates curves** (discount & forward)
- **Credit curves** (hazard/spread)
- **Inflation curves**
- **Correlations** (structured credit)
- **FX**
- **Volatility**
- **Cross-factor** interactions
- **Model parameters** (prepay, default, recovery, conversion)
- **Market scalars** (dividends, equity/commodity prices, inflation indices)
- **Residual** (unexplained)

## Imports

In [1]:
import json
from datetime import date

from finstack.core.market_data import DiscountCurve, HazardCurve, MarketContext
from finstack.valuations import (
    PnlAttribution,
    attribute_pnl,
    attribute_pnl_from_spec,
    default_attribution_metrics,
    default_waterfall_order,
    validate_attribution_json,
)

print("Imported attribution helpers.")
print("Default waterfall order:", default_waterfall_order())
print("Metrics-based metrics:", default_attribution_metrics()[:5], "...")

Imported attribution helpers.
Default waterfall order: ['Carry', 'RatesCurves', 'CreditCurves', 'InflationCurves', 'Correlations', 'Fx', 'Volatility', 'ModelParameters', 'MarketScalars']
Metrics-based metrics: ['theta', 'dv01', 'cs01', 'vega', 'delta'] ...


## Setup: instrument and two market snapshots

We'll attribute the P&L of a **5-year USD corporate bond** between two consecutive dates. Between T₀ and T₁:

- Discount rates shift **+5 bp** (rates sell-off)
- Credit spreads widen **+10 bp**
- One day of carry accrues

This lets us see carry, rates, and credit factors clearly separated.

In [2]:
AS_OF_T0 = "2025-01-15"
AS_OF_T1 = "2025-01-16"

bond_json = json.dumps(
    {
        "type": "bond",
        "spec": {
            "id": "CORP-5Y-USD",
            "notional": {"amount": "1000000", "currency": "USD"},
            "issue_date": "2024-01-15",
            "maturity": "2029-01-15",
            "cashflow_spec": {
                "Fixed": {
                    "coupon_type": "Cash",
                    "rate": 0.05,
                    "freq": {"count": 6, "unit": "months"},
                    "dc": "Thirty360",
                    "bdc": "following",
                    "calendar_id": "weekends_only",
                    "stub": "None",
                    "end_of_month": False,
                    "payment_lag_days": 0,
                }
            },
            "discount_curve_id": "USD-OIS",
            "credit_curve_id": "CORP-HAZARD",
            "call_put": None,
            "attributes": {"tags": [], "meta": {}},
            "pricing_overrides": {},
        },
    }
)
print("Instrument JSON ready:", len(bond_json), "chars")

Instrument JSON ready: 544 chars


**T0 valuation date (`AS_OF_T0 = 2025-01-15`):** This date is a **coupon date** on the demo bond (semi-annual from 2024-01-15). After the Rust carry fix, **carry** should include **PV roll-down plus realized cashflows**. You should see carry on the order of **~$139/day** (coupon accrual path) rather than the previous **−$24,888** artifact. **`credit_curves_pnl`** should now be **nonzero** when credit moves between T0 and T1, reflecting the **`Bond::value` hazard engine** fix isolating credit from rates.

In [3]:
base_t0 = date.fromisoformat(AS_OF_T0)
base_t1 = date.fromisoformat(AS_OF_T1)

# --- T0 market ---
mc_t0 = MarketContext()
disc_t0 = DiscountCurve(
    "USD-OIS", base_t0,
    [(0.0, 1.0), (0.5, 0.980), (1.0, 0.960), (2.0, 0.920),
     (3.0, 0.880), (5.0, 0.800), (10.0, 0.650)],
    day_count="act_365f",
)
haz_t0 = HazardCurve(
    "CORP-HAZARD", base_t0,
    [(0.5, 0.010), (1.0, 0.012), (2.0, 0.015),
     (3.0, 0.018), (5.0, 0.022), (10.0, 0.028)],
    day_count="act_365f",
)
mc_t0.insert(disc_t0)
mc_t0.insert(haz_t0)
market_t0_json = mc_t0.to_json()

# --- T1 market: +5 bp rates, +10 bp credit ---
mc_t1 = MarketContext()
disc_t1 = DiscountCurve(
    "USD-OIS", base_t1,
    [(0.0, 1.0), (0.5, 0.9797), (1.0, 0.9594), (2.0, 0.9189),
     (3.0, 0.8785), (5.0, 0.7978), (10.0, 0.6468)],
    day_count="act_365f",
)
haz_t1 = HazardCurve(
    "CORP-HAZARD", base_t1,
    [(0.5, 0.011), (1.0, 0.013), (2.0, 0.016),
     (3.0, 0.019), (5.0, 0.023), (10.0, 0.029)],
    day_count="act_365f",
)
mc_t1.insert(disc_t1)
mc_t1.insert(haz_t1)
market_t1_json = mc_t1.to_json()

print(f"T0 market: {len(market_t0_json)} chars")
print(f"T1 market: {len(market_t1_json)} chars")

T0 market: 1525 chars
T1 market: 1539 chars


## Running attribution

`attribute_pnl()` is the main entry point. Pass the instrument, two market snapshots, dates, and a method descriptor — and get back a `PnlAttribution` directly:

```python
attr = attribute_pnl(instrument_json, market_t0_json, market_t1_json,
                     as_of_t0, as_of_t1, method)
```

The `method` argument selects the methodology: `"Parallel"`, `{"Waterfall": [...]}`, `"MetricsBased"`, or `{"Taylor": {...}}`.

For power-user workflows that need the full JSON envelope round-trip, use `attribute_pnl_from_spec()` and `validate_attribution_json()` instead.

In [4]:
from functools import partial

run = partial(attribute_pnl, bond_json, market_t0_json, market_t1_json, AS_OF_T0, AS_OF_T1)
print("Ready — run('Parallel'), run({'Waterfall': [...]}) etc.")

Ready — run('Parallel'), run({'Waterfall': [...]}) etc.


## 1. Parallel attribution

Each factor is isolated independently: restore T₀ for one factor while keeping T₁ for all others. Cross-effects appear in the residual.

In [5]:
parallel = run("Parallel")

print(parallel.explain())
print()
print(f"Residual within tolerance: {parallel.residual_within_tolerance()}")
print(f"Method: {parallel.method}")
print(f"Repricings: {parallel.num_repricings}")

Total P&L: USD -26968.97
  ├─ Carry: USD -24888.12 (92.3%)
  │   ├─ Coupon Income: USD 0.00
  │   ├─ Theta: USD -24888.12
  ├─ Rates Curves: USD -2080.84 (7.7%)
  └─ Residual: USD 0.00 (-0.0%)

Residual within tolerance: True
Method: Parallel
Repricings: 6


In [6]:
parallel.to_dataframe()

,instrument_id,method,t0,t1,currency,total_pnl,carry,rates_curves_pnl,credit_curves_pnl,inflation_curves_pnl,correlations_pnl,fx_pnl,vol_pnl,cross_factor_pnl,model_params_pnl,market_scalars_pnl,residual,residual_pct,num_repricings
0,CORP-5Y-USD,Parallel,2025-01-15,2025-01-16,USD,-26968.973654,-24888.128647,-2080.845007,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,6


## 2. Waterfall attribution

Factors are applied sequentially. By construction, the sum of all factor P&Ls equals the total — the residual should be near zero.

The **order matters**: earlier factors absorb more cross-effects. The `default_waterfall_order()` provides a standard starting point.

In [7]:
order = default_waterfall_order()
print("Default waterfall order:", order)

Default waterfall order: ['Carry', 'RatesCurves', 'CreditCurves', 'InflationCurves', 'Correlations', 'Fx', 'Volatility', 'ModelParameters', 'MarketScalars']


In [8]:
waterfall_method = {"Waterfall": ["Carry", "RatesCurves", "CreditCurves", "Fx", "Volatility"]}

waterfall = run(waterfall_method)

print(waterfall.explain())
print()
print(f"Residual: {waterfall.residual:.4f} ({waterfall.residual_pct:.4f}%)")
print(f"Residual within tolerance: {waterfall.residual_within_tolerance()}")

Total P&L: USD -26968.97
  ├─ Carry: USD -24888.12 (92.3%)
  │   ├─ Coupon Income: USD 0.00
  │   ├─ Theta: USD -24888.12
  ├─ Rates Curves: USD -2080.84 (7.7%)
  └─ Residual: USD 0.00 (-0.0%)

Residual: 0.0000 (-0.0000%)
Residual within tolerance: True


### Comparing different orderings

Waterfall attribution is order-dependent. Let's see how swapping rates and credit changes the decomposition:

In [9]:
import pandas as pd

order_rates_first = {"Waterfall": ["Carry", "RatesCurves", "CreditCurves"]}
order_credit_first = {"Waterfall": ["Carry", "CreditCurves", "RatesCurves"]}

wf_rates = run(order_rates_first)
wf_credit = run(order_credit_first)

comparison = pd.DataFrame(
    {
        "Rates first": {
            "carry": wf_rates.carry,
            "rates_curves": wf_rates.rates_curves_pnl,
            "credit_curves": wf_rates.credit_curves_pnl,
            "residual": wf_rates.residual,
            "total": wf_rates.total_pnl,
        },
        "Credit first": {
            "carry": wf_credit.carry,
            "rates_curves": wf_credit.rates_curves_pnl,
            "credit_curves": wf_credit.credit_curves_pnl,
            "residual": wf_credit.residual,
            "total": wf_credit.total_pnl,
        },
    }
)
comparison

,Rates first,Credit first
carry,-24888.128647,-24888.128647
rates_curves,-2080.845007,-2080.845007
credit_curves,0.000000,0.000000
residual,0.000000,0.000000
total,-26968.973654,-26968.973654


## 3. Metrics-based attribution

Uses pre-computed first- and second-order sensitivities (Theta, DV01, CS01, Gamma, Convexity, etc.) to approximate the P&L decomposition. This is the fastest method but least accurate for large market moves.

In [10]:
metrics = run("MetricsBased")

print(metrics.explain())
print()
print(f"Residual: {metrics.residual:.4f} ({metrics.residual_pct:.4f}%)")
print(f"Repricings: {metrics.num_repricings}")

Total P&L: USD -26968.97
  ├─ Carry: USD -24888.12 (92.3%)
  │   ├─ Theta: USD -24888.12
  ├─ Rates Curves: USD -2088.14 (7.7%)
  └─ Residual: USD 7.30 (-0.0%)

Residual: 7.3008 (-0.0271%)
Repricings: 0


## 4. Taylor attribution

Computes first-order sensitivities via bump-and-reprice at T₀, then multiplies by observed market moves. Optionally includes second-order (gamma/convexity) terms.

Unlike waterfall, the decomposition is factor-additive and does not depend on application order.

In [11]:
taylor_config = {"Taylor": {"include_gamma": False, "rate_bump_bp": 1.0, "credit_bump_bp": 1.0, "vol_bump": 0.01}}

taylor = run(taylor_config)

print(taylor.explain())
print()
print(f"Residual: {taylor.residual:.4f} ({taylor.residual_pct:.4f}%)")
print(f"Repricings: {taylor.num_repricings}")

Total P&L: USD -26968.97
  ├─ Carry: USD -24888.12 (92.3%)
  │   ├─ Theta: USD -24888.12
  ├─ Rates Curves: USD -2090.65 (7.8%)
  └─ Residual: USD 9.81 (-0.0%)

Residual: 9.8144 (-0.0364%)
Repricings: 7


In [12]:
taylor_gamma_config = {"Taylor": {"include_gamma": True, "rate_bump_bp": 1.0, "credit_bump_bp": 1.0, "vol_bump": 0.01}}

taylor_gamma = run(taylor_gamma_config)

print("Taylor with gamma (second-order):")
print(taylor_gamma.explain())
print()
print(f"Residual without gamma: {taylor.residual:.4f} ({taylor.residual_pct:.2f}%)")
print(f"Residual with gamma:    {taylor_gamma.residual:.4f} ({taylor_gamma.residual_pct:.2f}%)")

Taylor with gamma (second-order):
Total P&L: USD -26968.97
  ├─ Carry: USD -24888.12 (92.3%)
  │   ├─ Theta: USD -24888.12
  ├─ Rates Curves: USD -2088.40 (7.7%)
  └─ Residual: USD 7.56 (-0.0%)

Residual without gamma: 9.8144 (-0.04%)
Residual with gamma:    7.5610 (-0.03%)


## Method comparison

Stack all four results side-by-side to compare factor decompositions and residual quality.

In [13]:
import pandas as pd

results = {"Parallel": parallel, "Waterfall": waterfall, "MetricsBased": metrics, "Taylor": taylor}

rows = []
for name, attr in results.items():
    rows.append(
        {
            "Method": name,
            "Total P&L": attr.total_pnl,
            "Carry": attr.carry,
            "Rates": attr.rates_curves_pnl,
            "Credit": attr.credit_curves_pnl,
            "FX": attr.fx_pnl,
            "Vol": attr.vol_pnl,
            "Cross": attr.cross_factor_pnl,
            "Residual": attr.residual,
            "Residual %": f"{attr.residual_pct:.2f}%",
            "Repricings": attr.num_repricings,
        }
    )

df = pd.DataFrame(rows).set_index("Method")
df

,Total P&L,Carry,Rates,Credit,FX,Vol,Cross,Residual,Residual %,Repricings
Method,,,,,,,,,,
Parallel,-26968.973654,-24888.128647,-2080.845007,0.0,0.0,0.0,0.0,0.000000,-0.00%,6
Waterfall,-26968.973654,-24888.128647,-2080.845007,0.0,0.0,0.0,0.0,0.000000,-0.00%,7
MetricsBased,-26968.973654,-24888.128647,-2088.145763,0.0,0.0,0.0,0.0,7.300756,-0.03%,0
Taylor,-26968.973654,-24888.128647,-2090.659448,0.0,0.0,0.0,0.0,9.814441,-0.04%,7


## JSON round-trip and validation

Attribution specs and results serialize to JSON for logging, replay, and cross-language use.

In [14]:
attr_json = parallel.to_json()
print("PnlAttribution JSON (first 500 chars):")
print(attr_json[:500], "...")
print()

roundtrip = PnlAttribution.from_json(attr_json)
print(f"Round-trip total_pnl match: {roundtrip.total_pnl == parallel.total_pnl}")
print(repr(roundtrip))

PnlAttribution JSON (first 500 chars):
{
  "total_pnl": {
    "amount": "-26968.9736539639998227357863",
    "currency": "USD"
  },
  "carry": {
    "amount": "-24888.1286471727071329951285",
    "currency": "USD"
  },
  "rates_curves_pnl": {
    "amount": "-2080.8450067912926897406578",
    "currency": "USD"
  },
  "credit_curves_pnl": {
    "amount": "0.0000000000000000000000",
    "currency": "USD"
  },
  "inflation_curves_pnl": {
    "amount": "0",
    "currency": "USD"
  },
  "correlations_pnl": {
    "amount": "0",
    "currenc ...

Round-trip total_pnl match: True
PnlAttribution(id="CORP-5Y-USD", method=Parallel, total_pnl=-26968.97 USD, residual_pct=-0.00%)


In [15]:
envelope = json.dumps({
    "schema": "finstack.attribution/1",
    "attribution": {
        "instrument": json.loads(bond_json),
        "market_t0": json.loads(market_t0_json),
        "market_t1": json.loads(market_t1_json),
        "as_of_t0": AS_OF_T0,
        "as_of_t1": AS_OF_T1,
        "method": "Parallel",
    },
})

canonical = validate_attribution_json(envelope)
print(f"Validated spec: {len(canonical)} chars")
print(canonical[:300], "...")

result_json = attribute_pnl_from_spec(envelope)
print(f"\nResult envelope: {len(result_json)} chars")

Validated spec: 5611 chars
{
  "schema": "finstack.attribution/1",
  "attribution": {
    "instrument": {
      "type": "bond",
      "spec": {
        "id": "CORP-5Y-USD",
        "notional": {
          "amount": "1000000",
          "currency": "USD"
        },
        "issue_date": "2024-01-15",
        "maturity": "2029- ...

Result envelope: 2754 chars


## Diagnostics and verbose output

`explain_verbose()` shows all factors including those with zero P&L, useful for debugging. Diagnostic `notes` capture warnings from the attribution engine.

In [16]:
print(parallel.explain_verbose())
print()
print("Notes:", parallel.notes)

Total P&L: USD -26968.97
  ├─ Carry: USD -24888.12 (92.3%)
  │   ├─ Coupon Income: USD 0.00
  │   ├─ Theta: USD -24888.12
  ├─ Rates Curves: USD -2080.84 (7.7%)
  ├─ Credit Curves: USD 0.00 (-0.0%)
  ├─ Inflation Curves: USD 0.00 (-0.0%)
  ├─ Correlations: USD 0.00 (-0.0%)
  ├─ FX: USD 0.00 (-0.0%)
  ├─ Vol: USD 0.00 (-0.0%)
  ├─ Cross-Factor: USD 0.00 (-0.0%)
  ├─ Model Params: USD 0.00 (-0.0%)
  ├─ Market Scalars: USD 0.00 (-0.0%)
  └─ Residual: USD 0.00 (-0.0%)

Notes: []


## Takeaways

- **`attribute_pnl(instrument, mkt_t0, mkt_t1, t0, t1, method)`** is the main entry point — pass instruments and market data directly, get a `PnlAttribution` back.
- **`attribute_pnl_from_spec(envelope_json)`** is the power-user variant for full JSON round-trip workflows.
- **`PnlAttribution`** wraps the result with property accessors, `explain()`, `to_dataframe()`, and JSON round-trip.
- **Parallel** is simplest and order-independent; **Waterfall** guarantees the sum constraint; **Metrics-based** is fastest; **Taylor** gives sensitivity decomposition.
- For large portfolios, use `PnlAttribution.to_dataframe()` and `pd.concat` to build a full book-level attribution table.
- All results serialize to JSON for logging and replay.